# A scheme, a growth scenario, and keeping the two apart

The loop notebook you used in this course held two things steady that you now control. It used one scheme file, the narrow one, and it worked out where the new jobs went from an elasticity you set rather than letting you place them.

Here you decide both. You choose which scheme file is written into the cost matrix, and you choose where the growth lands. The results then arrive in two panels:

1. one for what the scheme does
1. and one for what your allocation does,

so that the two contributions are never added into a single figure.

The scheme is the proposed Tyne and Wear Metro extension to Washington, written into the cost matrix by the same two files you read earlier. The growth is 14,863 jobs, four per cent of the study area's 371,585.

Two values to set, four runs to do, one file to download, and a closing section to read before you start writing. Work through the sections in order, from the top of the page to the bottom.


## Where the files are

JupyterLite runs inside your browser. Nothing is installed on your machine and you do not need administrator rights.

The notebook and its data arrived with the site, so there is nothing to download and nothing to upload. Open the file browser - the panel down the left-hand side, or the folder icon in the far-left sidebar if it is not showing - and you will find this arrangement already in place:

```
AE1-growth-scenarios/
    AE1-growth-scenarios.ipynb
    data/
        zones_msoa.csv
        trip_ends_msoa.csv
        cost_matrix_msoa.csv
        deprivation_msoa.csv
        zones_msoa.geojson
        scheme_cost_adjustments_narrow.csv
        scheme_cost_adjustments_broad.csv
```

Every path in the code below assumes it. The notebook sits at the top of the folder and the data sits one level under it, so moving either one will break the loading section.

**What happens to anything you change**

Because there is no server behind this, whatever you save goes into your browser's own storage rather than onto a network drive. That has two
consequences:

1. Anything you want to keep should be downloaded - right-click the file in the file browser and choose **Download**.
1. And if you clear your browsing data, or if your employer's IT policy clears it for you, your saved work goes with it.

This matters more here than it did in the teaching notebooks. The record file this notebook writes is the evidence your submission rests on, so download it after every run rather than at the end of the evening.

Do not edit the CSV files. If you want to try something out on them, duplicate one first and work on the copy.

**Getting back to the original**

Should you change the notebook and want the version you started with, use **Help > Clear Browser Data**. But read the warning it gives you before confirming. It removes everything you have stored on this site, for every notebook here, and it cannot be undone, so download anything you care about first.


## Checking the files are where you think they are

Run the cell below before anything else. It reports what it can see, which is faster than reading an error message later and guessing what went wrong.


In [ ]:
import os

DATA_FOLDER = "data"

expected = [
    "zones_msoa.csv",
    "trip_ends_msoa.csv",
    "cost_matrix_msoa.csv",
    "deprivation_msoa.csv",
    "zones_msoa.geojson",
    "scheme_cost_adjustments_narrow.csv",
    "scheme_cost_adjustments_broad.csv",
]

print("Looking in:", os.path.abspath(DATA_FOLDER))
print()

if not os.path.isdir(DATA_FOLDER):
    print("That folder does not exist yet.")
    print("Check the folder names and check where this notebook is saved.")
else:
    found = sorted(os.listdir(DATA_FOLDER))
    for name in expected:
        status = "found" if name in found else "MISSING"
        print(f"  {name:38s} {status}")

## Reading the scheme and the growth scenarios before running them

An assumption buried in notebook code cannot be audited. Anyone reviewing the work has to read Python to find what was assumed, and almost nobody does. So the scheme sits in two CSV files whose comment headers the cell below prints in full, and the growth scenarios are written out in words beside them rather than left for a reader to work out from the code that applies them.

Read all of it now.

* The narrow file connects the seven Washington MSOAs to the three zones holding the regional centres;
* the broad file connects the same seven origins to every zone in Tyne and Wear that already has a Metro or heavy rail station.

Both apply the same twelve minutes to every pair they list (which is a course assumption rather than a Nexus figure).

The growth scenarios are hypothetical. They mark out a range of places employment growth could go, running from evenly spread to entirely concentrated in one zone, and each was chosen to be defensible rather than to be a forecast. The third is deliberately extreme. None comes from an adopted plan, and saying so in your presentation is worth more than pretending otherwise.


In [ ]:
import pandas as pd

GROWTH_SCENARIOS = {
    "flat": "The 14,863 jobs are shared across all 145 zones in proportion to "
            "the jobs each already has. No zone changes its share of regional "
            "employment. Hypothetical, and the reference the other two are "
            "measured against.",
    "washington": "All 14,863 jobs go to the seven Washington MSOAs, shared "
                  "equally between them. Stands for a policy of concentrating "
                  "employment growth at the proposed stations. Hypothetical.",
    "washington_centre": "All 14,863 jobs go to Washington Town Centre & "
                         "Biddick alone, which holds 4,275 jobs in the base "
                         "year. A deliberate bracket rather than a proposal, "
                         "and not a forecast anybody has made.",
}


def print_header(path):
    with open(path, encoding="utf-8-sig") as handle:
        for line in handle:
            if not line.startswith("#"):
                break
            print(line.rstrip())


for tag in ["narrow", "broad"]:
    path = f"{DATA_FOLDER}/scheme_cost_adjustments_{tag}.csv"
    print("=" * 78)
    print_header(path)
    table = pd.read_csv(path, encoding="utf-8-sig", comment="#")
    print(f"  Rows: {len(table)}"
          f"   Distinct origins: {table['origin_id'].nunique()}"
          f"   Distinct destinations: {table['destination_id'].nunique()}")
    print(f"  Change applied: {table['gc_change_minutes'].min():.2f} to "
          f"{table['gc_change_minutes'].max():.2f} minutes")
    print()

print("=" * 78)
print("GROWTH SCENARIOS, ALL HYPOTHETICAL")
print("=" * 78)
for name, description in GROWTH_SCENARIOS.items():
    print(f"  {name}")
    words = description.split()
    line = "   "
    for word in words:
        if len(line) + len(word) + 1 > 74:
            print(line)
            line = "   "
        line += " " + word
    print(line)
    print()

## Parameters

This is the only cell in the notebook you will change. Everything below it reads these two values and does as it is told.

SCHEME_DEFINITION takes `"narrow"` or `"broad"` and names which scheme file is written into the cost matrix. It settles how far the modeller decided the extension's benefits reach.

GROWTH_ALLOCATION takes `"flat"`, `"washington"` or `"washington_centre"` and names where the 14,863 jobs go. On `"flat"` your run and the reference run are the same run, so the second panel will report zero throughout. That is the correct answer and it is worth seeing once.

Four values are fixed and none of them appears in the cell below:

* beta at 0.1185 per minute of generalised cost,
* the growth total at 14,863 jobs,
* the balancing tolerance at 0.01 trips,
* and the cap on balancing iterations at 200.

They are held still so that your runs can be compared. If beta could move as well, a difference between two runs might have come from the scheme file, from the growth allocation, from beta, or from all three at once, and nothing printed below would tell you which. With two values free and one changed at a time, you can say what caused what.


In [ ]:
# ---------------------------------------------------------------------------
# PARAMETERS
# ---------------------------------------------------------------------------

SCHEME_DEFINITION = "narrow"     # "narrow" or "broad"

GROWTH_ALLOCATION = "flat"       # "flat", "washington" or "washington_centre"

# ---------------------------------------------------------------------------

## Loading the data

Five files, of which the cost matrix is much the largest at 21,025 rows, so give the cell a moment before deciding it has stalled. The zone list fixes the order of everything else, so that the third zone in the cost matrix and the third entry in the jobs column are the same place.

Resident workers and jobs both total 371,585, which is not a duplication. Both are totals taken from the same 2011 commuting table, one summed across its rows and one down its columns, so each commuter is counted once where they live and once again where they work.

Population arrives from the deprivation file and is used for one comparison only, in the first panel. Nothing in this notebook allocates population or moves anybody's home.


In [ ]:
import numpy as np

BETA = 0.1185          # fixed, per minute of generalised cost
GROWTH_TOTAL = 14863   # fixed, jobs to be allocated
TOLERANCE = 0.01       # fixed, trips
MAX_ITERATIONS = 200   # fixed

zones = pd.read_csv(f"{DATA_FOLDER}/zones_msoa.csv", encoding="utf-8-sig")
trip_ends = pd.read_csv(f"{DATA_FOLDER}/trip_ends_msoa.csv",
                        encoding="utf-8-sig")
deprivation = pd.read_csv(f"{DATA_FOLDER}/deprivation_msoa.csv",
                          encoding="utf-8-sig")
costs = pd.read_csv(f"{DATA_FOLDER}/cost_matrix_msoa.csv",
                    encoding="utf-8-sig")

zone_ids = list(zones["zone_id"])
position = {zone: i for i, zone in enumerate(zone_ids)}

details = (zones[["zone_id", "zone_name", "local_authority"]]
           .merge(trip_ends[["zone_id", "resident_workers", "jobs"]],
                  on="zone_id")
           .merge(deprivation[["zone_id", "population"]], on="zone_id")
           .set_index("zone_id")
           .reindex(zone_ids))

zone_names = list(details["zone_name"])
base_jobs = details["jobs"].values.astype(float)
base_workers = details["resident_workers"].values.astype(float)
population = details["population"].values.astype(float)

base_cost = (costs
             .pivot(index="origin_id", columns="destination_id",
                    values="gc_min")
             .reindex(index=zone_ids, columns=zone_ids)
             .values.astype(float))

adjustments = pd.read_csv(
    f"{DATA_FOLDER}/scheme_cost_adjustments_{SCHEME_DEFINITION}.csv",
    encoding="utf-8-sig", comment="#")

scheme_cost = base_cost.copy()
for origin, destination, change in adjustments.itertuples(index=False):
    i, j = position[origin], position[destination]
    scheme_cost[i, j] = base_cost[i, j] + change

# The seven MSOAs covering the Washington townships. Both scheme files are
# symmetric, so their origin lists also contain the regional centres at the
# other end of each improved pair, which is why this set is stated here rather
# than read off the scheme file.
WASHINGTON = ["E02001797",   # Springwell & Usworth
              "E02001799",   # Concord and Sulgrave
              "E02001800",   # Albany and Blackfell
              "E02001807",   # Columbia, Barmston & Teal Farm
              "E02001809",   # Washington Town Centre & Biddick
              "E02001810",   # Oxclose and Lambton
              "E02001815"]   # Harraton, Rickleton & Fatfield
washington_rows = [position[zone] for zone in WASHINGTON]
TOWN_CENTRE = "E02001809"

print(f"Zones loaded:            {len(zone_ids)}")
print(f"Cost matrix:             {base_cost.shape[0]} by "
      f"{base_cost.shape[1]}   ({base_cost.size:,} ordered pairs)")
print(f"Resident workers:        {base_workers.sum():,.0f}")
print(f"Jobs, base year:         {base_jobs.sum():,.0f}")
print(f"Population, base year:   {population.sum():,.0f}")
print(f"Scheme file:             {SCHEME_DEFINITION}"
      f"   ({len(adjustments)} rows applied)")
print(f"Washington MSOAs:        {len(WASHINGTON)}"
      f"   holding {base_jobs[washington_rows].sum():,.0f} jobs")
for zone in WASHINGTON:
    print(f"                         {zone}  "
          f"{zone_names[position[zone]]}")
print(f"Growth allocation:       {GROWTH_ALLOCATION}")
print(f"Growth to allocate:      {GROWTH_TOTAL:,} jobs")
print(f"Beta:                    {BETA} per minute")

## Panel one: what the scheme does, with land use held still

This is the forecast you already know how to produce. The cost matrix changes, accessibility responds, and not one job moves in consequence. Its figures depend on SCHEME_DEFINITION and on nothing else, so they will be identical across every run you do with the same scheme file, whatever you set the growth allocation to.

That they do not move is the point of separating the panels. The scheme's own contribution is settled here, before your allocation is applied, so if you hold the scheme file steady across two runs, anything that moves between them came from your allocation and not from the scheme.

The last table asks the same question twice. It ranks the beneficiaries by accessibility to jobs, then by accessibility to population, using the same cost matrix and the same scheme file. Read the two orderings against each other before you go on.


In [ ]:
def balance(cost, origins, destinations):
    deterrence = np.exp(-BETA * cost)
    row_factor = np.ones(len(origins))
    col_factor = np.ones(len(destinations))
    for iteration in range(1, MAX_ITERATIONS + 1):
        row_factor = 1.0 / (deterrence * (col_factor * destinations)).sum(axis=1)
        col_factor = 1.0 / (deterrence
                            * (row_factor * origins)[:, None]).sum(axis=0)
        modelled = ((row_factor * origins)[:, None]
                    * (col_factor * destinations)[None, :]
                    * deterrence)
        row_error = np.abs(modelled.sum(axis=1) - origins).max()
        col_error = np.abs(modelled.sum(axis=0) - destinations).max()
        if max(row_error, col_error) < TOLERANCE:
            break
    return modelled, iteration


def mean_cost(matrix, cost):
    return (matrix * cost).sum() / matrix.sum()


matrix_dominimum, iters_dm = balance(base_cost, base_workers, base_jobs)
matrix_schemeonly, iters_so = balance(scheme_cost, base_workers, base_jobs)

gc_dominimum = mean_cost(matrix_dominimum, base_cost)
gc_schemeonly = mean_cost(matrix_schemeonly, scheme_cost)
scheme_saving = gc_dominimum - gc_schemeonly

access_jobs_dm = np.exp(-BETA * base_cost) @ base_jobs
access_jobs_sc = np.exp(-BETA * scheme_cost) @ base_jobs
benefit_jobs = 100.0 * (access_jobs_sc - access_jobs_dm) / access_jobs_dm

access_pop_dm = np.exp(-BETA * base_cost) @ population
access_pop_sc = np.exp(-BETA * scheme_cost) @ population
benefit_pop = 100.0 * (access_pop_sc - access_pop_dm) / access_pop_dm

panel_one = pd.DataFrame({
    "zone_id": zone_ids,
    "zone_name": zone_names,
    "local_authority": list(details["local_authority"]),
    "benefit_jobs_pct": benefit_jobs,
    "benefit_population_pct": benefit_pop,
})

print("PANEL ONE      THE SCHEME ALONE, LAND USE HELD AT THE BASE YEAR")
print(f"               scheme file: {SCHEME_DEFINITION}")
print("=" * 78)
print(f"Mean generalised cost, do-minimum:  {gc_dominimum:.4f} minutes"
      f"   ({iters_dm} balancing iterations)")
print(f"Mean generalised cost, scheme:      {gc_schemeonly:.4f} minutes"
      f"   ({iters_so} balancing iterations)")
print(f"Saving attributable to the scheme:  {scheme_saving:.4f} minutes"
      f" per trip")
print()
print(f"Zones gaining any accessibility to jobs: "
      f"{int((benefit_jobs > 1e-9).sum())} of {len(zone_ids)}")
print()
print("THE SAME SCHEME, RANKED TWICE")
print("-" * 78)
by_jobs = panel_one.nlargest(6, "benefit_jobs_pct")
by_pop = panel_one.nlargest(6, "benefit_population_pct")
print("Ranked by accessibility to jobs")
print(by_jobs[["zone_name", "local_authority", "benefit_jobs_pct"]]
      .round({"benefit_jobs_pct": 2}).to_string(index=False))
print()
print("Ranked by accessibility to population")
print(by_pop[["zone_name", "local_authority", "benefit_population_pct"]]
      .round({"benefit_population_pct": 2}).to_string(index=False))

## Where the growth goes

The 14,863 jobs now arrive. Your allocation puts them somewhere, and the cell below reports what that did to the distribution of employment across the 145 zones, against the flat allocation that shares them in proportion to the jobs each zone already has.

**A zone showing a negative figure in that comparison has not lost jobs.** It has received fewer of the new ones than it would have been given by a rule that shared them out in proportion to existing employment.


In [ ]:
weights = {
    "flat": base_jobs / base_jobs.sum(),
    "washington": np.array([1.0 if zone in WASHINGTON else 0.0
                            for zone in zone_ids]),
    "washington_centre": np.array([1.0 if zone == TOWN_CENTRE else 0.0
                                   for zone in zone_ids]),
}

share_run = weights[GROWTH_ALLOCATION] / weights[GROWTH_ALLOCATION].sum()
share_flat = weights["flat"] / weights["flat"].sum()

added_run = GROWTH_TOTAL * share_run
added_flat = GROWTH_TOTAL * share_flat

jobs_run = base_jobs + added_run
jobs_flat = base_jobs + added_flat
moved = added_run - added_flat
moved_total = float(np.clip(moved, 0.0, None).sum())

allocation = pd.DataFrame({
    "zone_id": zone_ids,
    "zone_name": zone_names,
    "base_jobs": base_jobs,
    "added": added_run,
    "revised_jobs": jobs_run,
    "vs_flat": moved,
})

print(f"WHERE THE {GROWTH_TOTAL:,} JOBS WENT"
      f"      allocation: {GROWTH_ALLOCATION}")
print("-" * 78)
print(f"Jobs relocated by your allocation: {moved_total:,.1f}"
      f"   ({100.0 * moved_total / GROWTH_TOTAL:.2f} per cent of the growth)")
print(f"Zones receiving more than flat:    "
      f"{int((moved > 1e-9).sum())}")
print(f"Zones receiving less than flat:    "
      f"{int((moved < -1e-9).sum())}")
print(f"Jobs in the seven Washington MSOAs: "
      f"{base_jobs[washington_rows].sum():,.0f} becomes "
      f"{jobs_run[washington_rows].sum():,.0f}")
print()

if moved_total > 1e-6:
    show = pd.concat([allocation.nlargest(6, "vs_flat"),
                      allocation.nsmallest(4, "vs_flat")])
    print("The six largest and four smallest changes against the flat "
          "allocation")
    print(show.round({"base_jobs": 0, "added": 1, "revised_jobs": 1,
                      "vs_flat": 1}).to_string(index=False))
else:
    print("Your allocation is the flat allocation, so nothing has moved.")
    print("That is the correct answer and it is worth seeing once.")

## Panel two: what your allocation does, with the scheme already in place

Both runs compared here hold 386,448 jobs and use the same scheme file, so the only thing left to separate them is where those jobs sit. The reference is the flat allocation rather than the do-minimum, because a comparison against the do-minimum would carry the scheme's effect and your allocation's effect together in one figure, with no way of saying how much of it was which.

Four quantities are reported:

* Mean generalised cost per modelled trip is stated as a give-back, meaning the share of the scheme's own saving from panel one that your allocation hands back.
* Self-containment is the share of trips beginning in the seven Washington MSOAs that also end there.
* The accessibility figures name Washington Town Centre & Biddick specifically, because it is the zone the campaign is about.
* And the balancing iteration count is printed for both runs, so that you can confirm each reached the tolerance of 0.01 trips rather than stopping at the cap of 200.

In [ ]:
workers_grown = base_workers * (jobs_run.sum() / base_workers.sum())

matrix_run, iters_run = balance(scheme_cost, workers_grown, jobs_run)
matrix_flat, iters_flat = balance(scheme_cost, workers_grown, jobs_flat)

gc_run = mean_cost(matrix_run, scheme_cost)
gc_flat = mean_cost(matrix_flat, scheme_cost)
giveback = gc_run - gc_flat
giveback_share = 100.0 * giveback / scheme_saving

access_run = np.exp(-BETA * scheme_cost) @ jobs_run
access_flat = np.exp(-BETA * scheme_cost) @ jobs_flat
benefit_run = 100.0 * (access_run - access_jobs_dm) / access_jobs_dm
benefit_flat = 100.0 * (access_flat - access_jobs_dm) / access_jobs_dm
shift = benefit_run - benefit_flat


def self_containment(matrix):
    inside = matrix[np.ix_(washington_rows, washington_rows)].sum()
    total = matrix[washington_rows, :].sum()
    return 100.0 * inside / total


contain_run = self_containment(matrix_run)
contain_flat = self_containment(matrix_flat)

town = position[TOWN_CENTRE]

panel_two = pd.DataFrame({
    "zone_id": zone_ids,
    "zone_name": zone_names,
    "local_authority": list(details["local_authority"]),
    "benefit_flat_pct": benefit_flat,
    "benefit_run_pct": benefit_run,
    "shift_pp": shift,
})

print("PANEL TWO      YOUR ALLOCATION AGAINST THE FLAT REFERENCE")
print(f"               scheme file: {SCHEME_DEFINITION}"
      f"      allocation: {GROWTH_ALLOCATION}")
print("=" * 78)
print(f"Jobs in both runs:                  {jobs_run.sum():,.0f}"
      f"   (identical, so the levels are comparable)")
print(f"Balancing iterations:               your run {iters_run},"
      f" flat reference {iters_flat}")
print()
print(f"Mean generalised cost, flat:        {gc_flat:.4f} minutes")
print(f"Mean generalised cost, your run:    {gc_run:.4f} minutes")
print(f"Given back by your allocation:      {giveback:+.4f} minutes per trip")
print(f"  as a share of the scheme saving:  {giveback_share:+.1f} per cent"
      f"   (scheme saved {scheme_saving:.4f} in panel one)")
print()
print(f"Washington self-containment, flat:  {contain_flat:.2f} per cent")
print(f"Washington self-containment, run:   {contain_run:.2f} per cent")
print(f"  shift:                            "
      f"{contain_run - contain_flat:+.2f} percentage points")
print()
print(f"Washington Town Centre & Biddick, accessibility benefit against the")
print(f"do-minimum:  flat {benefit_flat[town]:.2f} per cent,"
      f"  your run {benefit_run[town]:.2f} per cent,"
      f"  shift {shift[town]:+.2f} pp")
print()

if np.abs(shift).max() > 1e-6:
    print("Largest shifts against the flat reference")
    print("-" * 78)
    movers = pd.concat([panel_two.nlargest(6, "shift_pp"),
                        panel_two.nsmallest(4, "shift_pp")])
    print(movers[["zone_name", "local_authority", "benefit_flat_pct",
                  "benefit_run_pct", "shift_pp"]]
          .round({"benefit_flat_pct": 2, "benefit_run_pct": 2,
                  "shift_pp": 3}).to_string(index=False))
else:
    print("Your allocation is the flat allocation, so every shift is zero.")

## Mapping the two panels

Two maps, matching the two panels above:

* The left one is the scheme's effect with land use fixed, and it will not change between runs sharing a scheme file.
* The right one is your allocation's effect against the flat reference, and it is the only panel that responds to GROWTH_ALLOCATION.

How the colours were assigned decides what a map appears to show, so the rule is set out here rather than left to be inferred from the legend. Four steps:

1. Any zone whose value is exactly zero is put in a class by itself and shaded grey. On the left panel under the narrow file that is 135 of the 145 zones, because the scheme file names no pair on their row of the cost matrix.
2. The zones that did gain are sorted and cut into four groups of roughly equal size. Under the narrow file those groups hold 3, 2, 2 and 3 zones.
3. The cut points come from the gaining zones alone rather than from all 145. Under the narrow file they fall at 21.22, 41.52 and 45.93 per cent.
4. On the right panel only, a zone sitting below the flat reference is shaded in the second colour. An allocation that sends jobs towards Washington sends them away from everywhere else, and were those zones left on the same colour ramp as the gains they would come out as the palest gain, which is the reverse of what happened to them.

Quantiles across all 145 zones would be useless on the left panel. Under the narrow file 135 zones change by exactly nothing, so all four quintile break points would land on zero.



In [ ]:
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon, Patch
from matplotlib.collections import PatchCollection
from matplotlib.colors import LinearSegmentedColormap

with open(f"{DATA_FOLDER}/zones_msoa.geojson", encoding="utf-8-sig") as handle:
    boundaries = json.load(handle)

rings = {}
for feature in boundaries["features"]:
    geometry = feature["geometry"]
    parts = ([geometry["coordinates"]] if geometry["type"] == "Polygon"
             else geometry["coordinates"])
    rings[feature["properties"]["zone_id"]] = [part[0] for part in parts]

GREY = "#F0EFED"
RED = "#C8102E"
GREEN = "#025944"
INK = "#1A4538"
# Both panels show a change, so both use the same ramp for a gain. Green is
# reserved for the right panel's zones falling below the flat reference, and
# appears nowhere else, so one colour never means two things.
ramp_gain = LinearSegmentedColormap.from_list("gain", [GREY, RED])


def banded_gains(values, ramp):
    """Zones changing by nothing get their own class; the rest split at
    quartiles of the non-zero values. Used where every change is a gain."""
    values = np.asarray(values, dtype=float)
    live = values[values > 1e-6]
    if len(live) < 4:
        return [GREY] * len(values), []
    breaks = list(np.quantile(live, [0.25, 0.5, 0.75]))
    classes = 1 + np.searchsorted(breaks, values, side="right")
    classes[values <= 1e-6] = 0
    colours = [GREY if c == 0 else ramp(0.25 + 0.75 * c / 4) for c in classes]
    return colours, breaks


def banded_signed(values, ramp):
    """Same treatment for the gaining zones, with every zone doing worse than
    the reference given a single class of its own in the second colour. A one
    directional ramp would show a loss as a pale gain."""
    values = np.asarray(values, dtype=float)
    live = values[values > 1e-6]
    if len(live) < 4:
        return [GREY] * len(values), [], 0
    breaks = list(np.quantile(live, [0.25, 0.5, 0.75]))
    classes = 1 + np.searchsorted(breaks, values, side="right")
    classes[np.abs(values) <= 1e-6] = 0
    colours = []
    for value, klass in zip(values, classes):
        if value < -1e-6:
            colours.append(GREEN)
        elif klass == 0:
            colours.append(GREY)
        else:
            colours.append(ramp(0.25 + 0.75 * klass / 4))
    return colours, breaks, int((values < -1e-6).sum())


scheme_colours, scheme_breaks = banded_gains(benefit_jobs, ramp_gain)
growth_colours, growth_breaks, growth_losers = banded_signed(shift,
                                                              ramp_gain)

fig, axes = plt.subplots(1, 2, figsize=(9.6, 6.4))
titles = (f"Scheme alone, land use fixed ({SCHEME_DEFINITION})",
          f"Your allocation against flat ({GROWTH_ALLOCATION})")
for ax, colours, heading in zip(axes, (scheme_colours, growth_colours),
                                titles):
    patches, face, edge, width = [], [], [], []
    for zone, colour in zip(zone_ids, colours):
        for ring in rings[zone]:
            patches.append(MplPolygon(np.array(ring), closed=True))
            face.append(colour)
            edge.append(INK if zone in WASHINGTON else "white")
            width.append(1.1 if zone in WASHINGTON else 0.3)
    ax.add_collection(PatchCollection(patches, facecolors=face,
                                      edgecolors=edge, linewidths=width))
    ax.autoscale_view()
    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(heading, fontsize=9, color=INK)


def legend_for(ax, breaks, ramp, unit, title, losers=None):
    if not breaks:
        ax.legend(handles=[Patch(facecolor=GREY, edgecolor="white",
                                 label="no change anywhere")],
                  loc="upper left", fontsize=7, frameon=False,
                  title=title, title_fontsize=7)
        return
    handles = []
    if losers:
        handles.append(Patch(facecolor=GREEN, edgecolor="white",
                             label=f"below the reference ({losers} zones)"))
    handles.append(Patch(facecolor=GREY, edgecolor="white",
                         label="no change at all"))
    labels = [f"up to {breaks[0]:,.2f}{unit}",
              f"{breaks[0]:,.2f} to {breaks[1]:,.2f}{unit}",
              f"{breaks[1]:,.2f} to {breaks[2]:,.2f}{unit}",
              f"{breaks[2]:,.2f}{unit} and above"]
    handles += [Patch(facecolor=ramp(0.25 + 0.75 * (k + 1) / 4),
                      edgecolor="white", label=labels[k])
                for k in range(4)]
    ax.legend(handles=handles, loc="upper left", fontsize=7, frameon=False,
              title=title, title_fontsize=7)


legend_for(axes[0], scheme_breaks, ramp_gain, " pc",
           "Benefit, per cent of do-minimum")
legend_for(axes[1], growth_breaks, ramp_gain, " pp",
           "Shift against flat, percentage points", losers=growth_losers)

caption = (f"Tyne and Wear, 145 MSOAs. Scheme file {SCHEME_DEFINITION}, growth "
           f"allocation {GROWTH_ALLOCATION}, beta {BETA} per minute, "
           f"{GROWTH_TOTAL:,} jobs allocated. Left panel is accessibility to "
           "jobs under the scheme against the do-minimum, land use held at the "
           "base year. Right panel is the same measure under your allocation "
           "against the flat allocation, both holding "
           f"{jobs_run.sum():,.0f} jobs. Each panel gives zones changing by "
           "nothing their own class and splits the rest at quartiles of the "
           "non-zero values, so the two legends are not on a common scale. On the "
           "right panel every zone falling below the flat reference shares one "
           "class in the second colour. The seven Washington MSOAs are "
           "outlined in both panels.")
fig.text(0.06, 0.085, caption, ha="left", va="top", fontsize=7, color=INK,
         wrap=True)
fig.subplots_adjust(left=0.02, right=0.98, top=0.95, bottom=0.10, wspace=0.02)
plt.show()

## The run record

The cell below writes this run into `AE1-run-record.csv` in this folder, keyed on the scheme file and the growth allocation together, so running the same combination twice replaces its row rather than adding a second one.

Beside the outputs it carries the inputs, including beta and the growth total, which were fixed for you rather than chosen by you. Somebody who is handed a converged accessibility surface for Washington cannot tell from the numbers alone how wide the scheme was drawn or where the growth was put, and both of those change the answer. A run with no record of its parameters is not evidence.

Download this file after every run. It is what your submission has to reproduce.


In [ ]:
RECORD = "AE1-run-record.csv"

entry = {
    "run": f"{SCHEME_DEFINITION}_{GROWTH_ALLOCATION}",
    "scheme_definition": SCHEME_DEFINITION,
    "growth_allocation": GROWTH_ALLOCATION,
    "beta": BETA,
    "growth_total": GROWTH_TOTAL,
    "opportunity": "jobs",
    "scheme_rows_applied": len(adjustments),
    "gc_dominimum_min": round(float(gc_dominimum), 4),
    "gc_scheme_only_min": round(float(gc_schemeonly), 4),
    "scheme_saving_min": round(float(scheme_saving), 4),
    "zones_gaining_scheme": int((benefit_jobs > 1e-9).sum()),
    "top_by_jobs": zone_names[int(np.argmax(benefit_jobs))],
    "top_by_jobs_pct": round(float(benefit_jobs.max()), 2),
    "top_by_population": zone_names[int(np.argmax(benefit_pop))],
    "top_by_population_pct": round(float(benefit_pop.max()), 2),
    "gc_flat_min": round(float(gc_flat), 4),
    "gc_run_min": round(float(gc_run), 4),
    "giveback_min": round(float(giveback), 4),
    "giveback_pct_of_scheme_saving": round(float(giveback_share), 1),
    "jobs_relocated": round(moved_total, 1),
    "washington_jobs_after": round(float(jobs_run[washington_rows].sum()), 0),
    "self_containment_flat_pct": round(float(contain_flat), 2),
    "self_containment_run_pct": round(float(contain_run), 2),
    "self_containment_shift_pp": round(float(contain_run - contain_flat), 2),
    "town_centre_benefit_flat_pct": round(float(benefit_flat[town]), 2),
    "town_centre_benefit_run_pct": round(float(benefit_run[town]), 2),
    "town_centre_shift_pp": round(float(shift[town]), 3),
    "balancing_iterations_run": iters_run,
    "balancing_iterations_flat": iters_flat,
}

record = pd.DataFrame([entry])
if os.path.exists(RECORD):
    previous = pd.read_csv(RECORD, encoding="utf-8-sig")
    previous = previous[previous["run"] != entry["run"]]
    record = pd.concat([previous, record], ignore_index=True)

record = record.sort_values(["scheme_definition", "growth_allocation"])
record = record.reset_index(drop=True)
record.to_csv(RECORD, index=False, encoding="utf-8-sig")

panel_two.round(4).to_csv("AE1-accessibility-by-zone.csv", index=False,
                          encoding="utf-8-sig")

print("RUN RECORD")
print("=" * 78)
for column in record.columns:
    values = "   ".join(str(value) for value in record[column])
    print(f"  {column:32s} {values}")
print()
print(f"Runs recorded so far: {len(record)}")
print()
print("Written: AE1-run-record.csv")
print("Written: AE1-accessibility-by-zone.csv")
print("Right-click each one in the file browser and choose Download.")

## What to do now

Four runs. Use **Kernel > Restart Kernel and Run All Cells** each time, and download `AE1-run-record.csv` after each one.

1. SCHEME_DEFINITION `"narrow"`, GROWTH_ALLOCATION `"flat"`. This is the reference. Confirm that panel two reports zero throughout.
2. SCHEME_DEFINITION `"narrow"`, GROWTH_ALLOCATION `"washington"`.
3. SCHEME_DEFINITION `"broad"`, GROWTH_ALLOCATION `"flat"`.
4. One further combination of your own choosing. Two values with three settings and two settings give six combinations altogether, and the runs above have used three, so you are choosing between: 1) "broad" with "washington", 2) "narrow" with "washington_centre", or 3) "broad" with "washington_centre"

Before you look at the record file as a whole, write down what you expect the fourth run to show. Comparing that against what it did show is worth a slide.


## Before you write

Four things to have in front of you, and none of them is an interpretation. What the runs mean is the part you are being assessed on, so this section gives you the arithmetic and stops there.

**The figures your record file holds**

Panel one gives the scheme's own contribution:

* the two mean generalised costs,
* the saving between them,
* the count of zones gaining anything,
* and the two beneficiary rankings.

Panel two gives your allocation's contribution:

* the give-back in minutes and as a share,
* the jobs your allocation relocated,
* the shift in self-containment,
* and the shift in Washington Town Centre & Biddick's benefit.

Presenting a panel one figure as though your allocation had produced it would misattribute the whole finding.

**What the give-back is a share of**

It is expressed against the scheme saving computed in panel one, under whichever scheme file that run used. That denominator is not constant. It is 0.1428 minutes under the narrow file and 0.3861 under the broad one, so a give-back from a narrow run and a give-back from a broad run are shares of different quantities, and putting the two figures side by side without saying so would mislead a reader. A give-back above 100 per cent means your allocation handed back more than the scheme saved.

**What this model does not do**

Land use here does not respond to anything. You set it, no accessibility figure feeds back into where the jobs went, and so there is no loop in this notebook and nothing converges towards an equilibrium. Resident workers are grown by one factor applied to every zone, which holds the
existing pattern of where people live and asserts nothing about anybody moving house. The study area carries a single cost matrix of highway free-flow generalised cost, so a public transport scheme has been written in as though the road cost fell.

**What is assumed rather than observed**

The twelve-minute reduction, the pair lists in both scheme files, all three growth allocations, and the four per cent growth rate. Less obviously, the cost matrix as well. Its free-flow speeds, its value of time of twelve pounds an hour, its vehicle operating cost of fifteen pence a kilometre and its intrazonal rule of half the cost of reaching the nearest other zone were all set for this course rather than taken from a Transport Analysis Guidance release, as the dataset's README records. The 371,585 jobs are observed. They come from the Office for National Statistics 2011 commuting table WU03EW, summed to give each zone its workplace total.
